# UTKFace Data Preparation

This notebook prepares the UTKFace dataset by extracting age, gender, and ethnicity information from image filenames. It also fixes the image path issue by using `pathlib.Path` instead of string concatenation. Images are loaded from `project/data/UTKFace`.

## Data Preparation: Extracting Age, Gender, and Ethnicity from Image Filenames

Each filename in the UTKFace dataset follows this naming convention:

```text
age_gender_ethnicity_date.jpg
```

For example, the filename `25_1_3_20170116174525125.jpg.chip.jpg` contains these labels:

- `25` represents the age.
- `1` represents the gender code.
- `3` represents the ethnicity/race code.

The race/ethnicity codes are encoded as integers from `0` to `4`. In the code below, the gender mapping is set to `0 = Male` and `1 = Female`, which is the commonly used UTKFace convention. If your dataset source specifies `1 = Male` and `0 = Female`, swap the two labels in `GENDER_MAP` before running the notebook.

The preparation process is:

1. Set a fixed random seed for reproducible train/validation/test splits.
2. Read and shuffle image filenames.
3. Parse each filename by splitting on underscores (`_`).
4. Extract age, gender, and ethnicity labels.
5. Store the full image path using `Path`, avoiding broken paths such as `UTKFacefilename.jpg`.
6. Build a metadata table for later training and fairness evaluation.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, random_split
from torchvision import transforms

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

cwd = Path.cwd().resolve()
search_roots = [cwd, *cwd.parents]
UTKFACE_CANDIDATES = []
for root in search_roots:
    UTKFACE_CANDIDATES.extend([
        root / "project" / "data" / "UTKFace",
        root / "data" / "UTKFace",
    ])

UTKFACE_DIR = next((path for path in UTKFACE_CANDIDATES if path.exists()), UTKFACE_CANDIDATES[0])
DATA_DIR = UTKFACE_DIR.parent
PROJECT_ROOT = DATA_DIR.parent

print("Project root:", PROJECT_ROOT)
print("UTKFace dir:", UTKFACE_DIR)
print("UTKFace dir exists:", UTKFACE_DIR.exists())
if not UTKFACE_DIR.exists():
    print("Checked paths:")
    for path in UTKFACE_CANDIDATES[:8]:
        print(" -", path)

Project root: D:\work\extra\diversification\project
UTKFace dir: D:\work\extra\diversification\project\data\UTKFace
UTKFace dir exists: True


In [2]:
RACE_MAP = {
    0: "White",
    1: "Black",
    2: "Asian",
    3: "Indian",
    4: "Others",
}

GENDER_MAP = {
    0: "Male",
    1: "Female",
}

AGE_GROUPS = ["0-19", "20-39", "40-59", "60+"]

def age_group(age):
    if age <= 19:
        return "0-19"
    if age <= 39:
        return "20-39"
    if age <= 59:
        return "40-59"
    return "60+"

def parse_utkface_filename(path):
    parts = path.name.split("_")
    if len(parts) < 4:
        return None
    try:
        age = int(parts[0])
        gender = int(parts[1])
        race = int(parts[2])
    except ValueError:
        return None
    if gender not in GENDER_MAP or race not in RACE_MAP:
        return None
    return {
        "filename": path.name,
        "image_path": str(path),
        "age": age,
        "age_group": age_group(age),
        "gender": GENDER_MAP[gender],
        "gender_id": gender,
        "race": RACE_MAP[race],
        "race_id": race,
        "race_age_class": race * len(AGE_GROUPS) + AGE_GROUPS.index(age_group(age)),
    }

In [3]:
EXPECTED_COLUMNS = [
    "filename",
    "image_path",
    "age",
    "age_group",
    "gender",
    "gender_id",
    "race",
    "race_id",
    "race_age_class",
]

records = []
bad_files = []

image_paths = sorted(UTKFACE_DIR.glob("*.jpg"))
if not image_paths:
    raise FileNotFoundError(
        f"No .jpg files found in {UTKFACE_DIR}. "
        "Run the first cell and check that UTKFace dir exists is True."
    )

image_paths = list(rng.permutation(image_paths))

for image_path in image_paths:
    record = parse_utkface_filename(image_path)
    if record is None:
        bad_files.append(image_path.name)
    else:
        records.append(record)

df = pd.DataFrame(records, columns=EXPECTED_COLUMNS)
if df.empty:
    raise ValueError("No valid UTKFace filenames were parsed. Check the filename format.")
print("Parsed images:", len(df))
print("Skipped files:", len(bad_files))
df.head()

Parsed images: 23705
Skipped files: 3


,filename,image_path,age,age_group,gender,gender_id,race,race_id,race_age_class
0,46_0_1_20170113184447960.jpg.chip.jpg,D:\work\extra\diversification\project\data\UTK...,46,40-59,Male,0,Black,1,6
1,4_1_1_20170109194523891.jpg.chip.jpg,D:\work\extra\diversification\project\data\UTK...,4,0-19,Female,1,Black,1,4
2,68_1_1_20170110183842657.jpg.chip.jpg,D:\work\extra\diversification\project\data\UTK...,68,60+,Female,1,Black,1,7
3,35_1_1_20170112215312112.jpg.chip.jpg,D:\work\extra\diversification\project\data\UTK...,35,20-39,Female,1,Black,1,5
4,24_1_0_20170117194916850.jpg.chip.jpg,D:\work\extra\diversification\project\data\UTK...,24,20-39,Female,1,White,0,1


In [4]:
missing = df[~df["image_path"].map(lambda p: Path(p).exists())]
print("Missing image paths:", len(missing))

if len(missing):
    display(missing.head())
else:
    print("All image paths are valid.")

Missing image paths: 0
All image paths are valid.


In [5]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class UTKFaceDataset(Dataset):
    def __init__(self, metadata, transform=None):
        self.metadata = metadata.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        image_path = Path(row["image_path"])

        # This is the important fix: image_path is already a valid full path.
        # Do not build paths with strings like DATA_DIR + filename.
        image = Image.open(image_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        label = torch.zeros(20, dtype=torch.float32)
        label[int(row["race_age_class"])] = 1.0
        return image, label

dataset = UTKFaceDataset(df, transform=transform)
len(dataset)

23705

In [6]:
train_size = int(0.70 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

generator = torch.Generator().manual_seed(42)
train_data, val_data, test_data = random_split(dataset, [train_size, val_size, test_size], generator=generator)

len(train_data), len(val_data), len(test_data)

(16593, 3555, 3557)

In [7]:
classes = np.zeros(20, dtype=int)

for _, y in tqdm(train_data):
    classes[y.numpy() == 1] += 1

class_names = [f"{race}-{age_group}" for race in RACE_MAP.values() for age_group in AGE_GROUPS]
class_counts = pd.DataFrame({"class": class_names, "count": classes})
class_counts

100%|██████████| 16593/16593 [01:27<00:00, 189.10it/s]


,class,count
0,White-0-19,1409
1,White-20-39,2601
2,White-40-59,1759
3,White-60+,1240
4,Black-0-19,241
5,Black-20-39,2215
6,Black-40-59,516
7,Black-60+,235
8,Asian-0-19,716
9,Asian-20-39,1277


In [8]:
if "df" not in globals() or df.empty:
    raise ValueError("df is empty. Run the filename parsing cells first.")

DATA_DIR = UTKFACE_DIR.parent
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df = df.copy()

# Create train/val/test split
rng = np.random.default_rng(42)
indices = rng.permutation(len(df))

train_end = int(0.70 * len(df))
val_end = int(0.85 * len(df))

df["split"] = "test"
df.loc[indices[:train_end], "split"] = "train"
df.loc[indices[train_end:val_end], "split"] = "val"

csv_path = PROCESSED_DIR / "utkface_metadata.csv"
df.to_csv(csv_path, index=False)

print("Saved CSV:", csv_path)
print(df["split"].value_counts())
df.head()

Saved CSV: D:\work\extra\diversification\project\data\processed\utkface_metadata.csv
split
train    16593
val       3556
test      3556
Name: count, dtype: int64


,filename,image_path,age,age_group,gender,gender_id,race,race_id,race_age_class,split
0,46_0_1_20170113184447960.jpg.chip.jpg,D:\work\extra\diversification\project\data\UTK...,46,40-59,Male,0,Black,1,6,val
1,4_1_1_20170109194523891.jpg.chip.jpg,D:\work\extra\diversification\project\data\UTK...,4,0-19,Female,1,Black,1,4,val
2,68_1_1_20170110183842657.jpg.chip.jpg,D:\work\extra\diversification\project\data\UTK...,68,60+,Female,1,Black,1,7,train
3,35_1_1_20170112215312112.jpg.chip.jpg,D:\work\extra\diversification\project\data\UTK...,35,20-39,Female,1,Black,1,5,val
4,24_1_0_20170117194916850.jpg.chip.jpg,D:\work\extra\diversification\project\data\UTK...,24,20-39,Female,1,White,0,1,train


In [9]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

class UTKFaceClassificationDataset(Dataset):
    def __init__(self, metadata, transform=None):
        self.metadata = metadata.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        label = int(row["race_age_class"])  # 20-class label
        return image, label


train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

train_ds = UTKFaceClassificationDataset(df[df["split"] == "train"], transform=train_transform)
val_ds = UTKFaceClassificationDataset(df[df["split"] == "val"], transform=eval_transform)
test_ds = UTKFaceClassificationDataset(df[df["split"] == "test"], transform=eval_transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

print(len(train_ds), len(val_ds), len(test_ds))

16593 3556 3556


In [10]:
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet18

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

num_classes = 20

model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

def run_epoch(loader, training=True):
    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            outputs = model(images)
            loss = criterion(outputs, labels)

            if training:
                loss.backward()
                optimizer.step()

        preds = outputs.argmax(dim=1)

        total_loss += loss.item() * images.size(0)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


epochs = 20

for epoch in range(epochs):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss, val_acc = run_epoch(val_loader, training=False)

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"train loss: {train_loss:.4f}, train acc: {train_acc:.4f} | "
        f"val loss: {val_loss:.4f}, val acc: {val_acc:.4f}"
    )

Device: cuda
Epoch 1/20 | train loss: 1.8411, train acc: 0.4121 | val loss: 1.5680, val acc: 0.4823
Epoch 2/20 | train loss: 1.3656, train acc: 0.5460 | val loss: 1.7317, val acc: 0.4485
Epoch 3/20 | train loss: 1.1778, train acc: 0.6010 | val loss: 1.3561, val acc: 0.5495
Epoch 4/20 | train loss: 1.0412, train acc: 0.6480 | val loss: 1.4227, val acc: 0.5211
Epoch 5/20 | train loss: 0.9333, train acc: 0.6836 | val loss: 1.2330, val acc: 0.5855
Epoch 6/20 | train loss: 0.8158, train acc: 0.7240 | val loss: 1.2709, val acc: 0.5900
Epoch 7/20 | train loss: 0.7110, train acc: 0.7600 | val loss: 1.3633, val acc: 0.5593
Epoch 8/20 | train loss: 0.6007, train acc: 0.7968 | val loss: 1.3738, val acc: 0.5875
Epoch 9/20 | train loss: 0.5059, train acc: 0.8311 | val loss: 1.4161, val acc: 0.5861
Epoch 10/20 | train loss: 0.4239, train acc: 0.8616 | val loss: 1.2801, val acc: 0.5914
Epoch 11/20 | train loss: 0.3607, train acc: 0.8839 | val loss: 1.3824, val acc: 0.5990
Epoch 12/20 | train loss: 0.